# YOLOv13-S Baseline - AFB / Tuberculosis6208 (Chen split)

Mirror struktur `yolo12.ipynb` (wavelet-yolo12) supaya **apples-to-apples** vs YOLOv12s baseline.

**Setup:**
- Dataset zip di Drive: `MyDrive/Tuberculosis6208.zip` (Pascal-VOC).
- Split: **Chen et al. IJAI 2024** - 1024/140/101, `SPLIT_SEED=42` deterministic.
- Training: 1 run, `MODEL=yolov13s`, `SEED=42`, 60 epoch.
- Logging: **W&B** - project `afb_yolov13_chen`.
- Runtime: A100 ~ 25-30 menit per run.

Notebook portable Colab + local Windows (cells Colab-only auto-skip jika tidak terdeteksi).

## 0. Environment detection

In [ ]:
import sys
IS_COLAB = 'google.colab' in sys.modules
print('Environment :', 'Colab' if IS_COLAB else 'local')

## 1. Mount Drive (Colab only)

In [ ]:
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('skip: not Colab')

## 2. Clone repo (afb-yolo13) + YOLOv13 fork

**Colab:** clone fresh ke `/content/`. Cleanup cache supaya tidak konflik dgn previous run.

**Local:** asumsi `D:/Project/afb-yolo13` dan `D:/Project/yolov13` sudah ada.

In [ ]:
import os, gc
from pathlib import Path
import torch

if IS_COLAB:
    REPO_DIR    = Path('/content/afb-yolo13')
    YOLOV13_DIR = Path('/content/yolov13')

    # Cleanup caches (mirror yolo12.ipynb)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    !find /content -type d -name '__pycache__' -exec rm -rf {} + 2>/dev/null
    !find /content -type f -name '*.pyc' -delete 2>/dev/null
    !pip cache purge -q
    !rm -rf ~/.cache/ultralytics ~/.config/Ultralytics /root/.cache 2>/dev/null

    # afb-yolo13 (scripts + notebook)
    if REPO_DIR.exists():
        !cd {REPO_DIR} && git fetch origin && git checkout main && git pull --ff-only
    else:
        !git clone https://github.com/iswantosan/afb-yolo13.git {REPO_DIR}

    # YOLOv13 fork (iMoonLab)
    if YOLOV13_DIR.exists():
        !cd {YOLOV13_DIR} && git pull --ff-only
    else:
        !git clone https://github.com/iMoonLab/yolov13.git {YOLOV13_DIR}

    os.chdir(YOLOV13_DIR)
    sys.path.insert(0, str(YOLOV13_DIR))
    sys.path.insert(0, str(REPO_DIR))
    print('cwd:', os.getcwd())
    !cd {YOLOV13_DIR} && git log -1 --oneline
else:
    REPO_DIR    = Path('D:/Project/afb-yolo13')
    YOLOV13_DIR = Path('D:/Project/yolov13')
    print('Local repos:')
    print('  REPO_DIR   :', REPO_DIR, '(exists)' if REPO_DIR.exists() else '(MISSING)')
    print('  YOLOV13_DIR:', YOLOV13_DIR, '(exists)' if YOLOV13_DIR.exists() else '(MISSING)')

## 3. Install dependencies + apply L3 patches

Patch script copies custom AFB modules (RodDSC3k2, SpatialFullPAD_Tunnel, HyperACEScale, etc.) ke YOLOv13 source tree. Idempotent — skip kalau sudah ter-patch. Buat backup `.orig` di first apply.

In [ ]:
if IS_COLAB:
    # 1. Apply AFB-YOLOv13 patches FIRST (before pip install -e)
    !python {REPO_DIR}/scripts/apply_yolov13_patches.py {YOLOV13_DIR}
    # 2. Install YOLOv13 editable + wandb
    !pip -q install -e {YOLOV13_DIR} wandb
    !pip -q install -r {REPO_DIR}/requirements.txt
else:
    print('Local: pastikan sudah jalankan:')
    print(f'  python {REPO_DIR}/scripts/apply_yolov13_patches.py {YOLOV13_DIR}')
    print(f'  pip install -e {YOLOV13_DIR}')
    print(f'  pip install -r {REPO_DIR}/requirements.txt')

## 4. Build Chen split (1024 / 140 / 101, seed=42)

Extract zip (Colab) atau pakai dataset lokal (Windows). Skip jika output sudah ada.

In [ ]:
if IS_COLAB:
    DRIVE_ZIP   = '/content/drive/MyDrive/Tuberculosis6208.zip'
    EXTRACT_DIR = '/content/dataset/raw'
    DATASET_SRC = f'{EXTRACT_DIR}/tuberculosis-phonecamera'
    SPLIT_DIR   = '/content/tb_chen_split'
else:
    DRIVE_ZIP   = None
    DATASET_SRC = 'D:/project/yolov12/Tuberculosis6208/tuberculosis-phonecamera'
    SPLIT_DIR   = 'D:/datasets/tb_chen_split'

DATA_YAML = f'{SPLIT_DIR}/data.yaml'

if not Path(DATA_YAML).exists():
    cmd_parts = [
        'python', f'{REPO_DIR}/scripts/build_chen_split.py',
        '--src', f'"{DATASET_SRC}"',
        '--out', f'"{SPLIT_DIR}"',
        '--seed', '42',
    ]
    if DRIVE_ZIP and Path(DRIVE_ZIP).exists():
        cmd_parts += ['--zip', f'"{DRIVE_ZIP}"', '--extract-dir', f'"{EXTRACT_DIR}"']
    cmd = ' '.join(cmd_parts)
    print(cmd, '\n')
    os.system(cmd)
else:
    print(f'Split already exists at {SPLIT_DIR}')

print('\n--- data.yaml ---')
print(Path(DATA_YAML).read_text())

## 5. W&B login

In [ ]:
import wandb
wandb.login()

## 6. Config run

Ganti `MODEL_CFG` untuk varian:

**Baseline (Ultralytics stock):**
- `yolov13n.yaml` / `yolov13s.yaml` / `yolov13l.yaml` / `yolov13x.yaml`

**L1 hyperparam tweak (bukan novelty):**

| YAML | Strategy | Status |
|---|---|---|
| `yolov13s-fine.yaml` | head DSC3k2 k2=5 | 🟡 |
| `yolov13s-he16.yaml` | HyperACE num_hyperedges 8→16 | 🟡 |

**L2 connectivity (risky):**

| YAML | Status |
|---|---|
| `yolov13s-p2.yaml` | ✗ broke gates #6/#7 |

**L3 new module (engineering):**

| YAML | Module |
|---|---|
| `yolov13s-rod.yaml` | RodDSC3k2 |
| `yolov13s-spgate.yaml` | SpatialFullPAD_Tunnel |
| `yolov13s-scfuse.yaml` | HyperACEScale |

**L4 — HyperMIL (real novelty, image-level supervision):**

| Aktivasi | Cara |
|---|---|
| `USE_HYPERMIL = True` di cell ini + base yaml apa pun | Tambah aux MIL head + count loss tanpa ubah YAML |

HyperMIL: hypergraph-backed MIL aux head reading from HyperACE output. Target: label noise (66% far FP yang ternyata mostly real bacilli). Image-level count loss memaksa model discover all bacilli, bukan hanya GT-marked subset.

Run name auto = `<stem>_seed<S>_<EP>ep`, plus `_mil` suffix kalau HyperMIL aktif.

In [ ]:
# Pilih satu (uncomment yang mau dijalankan):
# === baseline ===
MODEL_CFG = 'yolov13s.yaml'                                         # baseline
# === L1 / L3 variants ===
# MODEL_CFG = str(REPO_DIR / 'configs' / 'yolov13s-fine.yaml')
# MODEL_CFG = str(REPO_DIR / 'configs' / 'yolov13s-he16.yaml')
# MODEL_CFG = str(REPO_DIR / 'configs' / 'yolov13s-rod.yaml')
# MODEL_CFG = str(REPO_DIR / 'configs' / 'yolov13s-spgate.yaml')
# MODEL_CFG = str(REPO_DIR / 'configs' / 'yolov13s-scfuse.yaml')

# === L4 HyperMIL toggle ===
USE_HYPERMIL    = False       # set True to enable HyperMIL aux head + count loss
MIL_WEIGHT      = 0.5         # weight for MIL count loss term
MIL_HIDDEN      = 128         # hidden dim for attention pooling + MLP
CONSIST_WEIGHT  = 0.0         # detection-MIL consistency regularizer (0 = off)

PRETRAINED    = 'yolov13s.pt'        # selalu yolov13s.pt (auto-download iMoonLab)
SEED          = 42
EPOCHS        = 60
IMGSZ         = 640
BATCH         = 16
DEVICE        = 0

WANDB_PROJECT = 'afb_yolov13_chen'
RUN_PROJECT   = '/content/runs/afb_yolov13' if IS_COLAB else 'D:/runs/afb_yolov13'
RUN_NAME      = f"{Path(MODEL_CFG).stem}_seed{SEED}_{EPOCHS}ep"
if USE_HYPERMIL:
    RUN_NAME += f'_mil{MIL_WEIGHT:g}'
    if CONSIST_WEIGHT > 0:
        RUN_NAME += f'_c{CONSIST_WEIGHT:g}'

print('cfg          :', MODEL_CFG)
print('seed         :', SEED)
print('epochs       :', EPOCHS)
print('USE_HYPERMIL :', USE_HYPERMIL, f'(weight={MIL_WEIGHT}, hidden={MIL_HIDDEN})' if USE_HYPERMIL else '')
print('run_name     :', RUN_NAME)
print('project      :', RUN_PROJECT)

## 7. Seed + SDP kernel + W&B init

In [ ]:
import random, numpy as np

# Stable SDP kernel (mirror yolo12.ipynb - avoid Flash/MEM-efficient mismatch)
os.environ['PYTORCH_SDP_KERNEL'] = 'math'
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_math_sdp(True)

# Reproducibility
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

# Disable Ultralytics built-in W&B callback (we log manually)
from ultralytics.utils import SETTINGS
SETTINGS.update({'wandb': False})

run = wandb.init(
    project=WANDB_PROJECT,
    name=RUN_NAME,
    reinit=True,
    config=dict(
        model_cfg=MODEL_CFG, data_yaml=DATA_YAML, pretrained=PRETRAINED,
        seed=SEED, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
        optimizer='SGD', lr0=0.01, momentum=0.937, cos_lr=True,
        split='chen_1024_140_101', split_seed=42,
    ),
    tags=[Path(MODEL_CFG).stem, f'seed{SEED}', 'chen_split', 'baseline_v13'],
)
print('W&B run:', run.url)

## 8. Auto-download pretrained yolov13s.pt

In [ ]:
from urllib.request import urlretrieve

pt = Path(PRETRAINED)
if not pt.exists():
    url = f'https://github.com/iMoonLab/yolov13/releases/download/yolov13/{PRETRAINED}'
    print(f'Downloading {url}')
    urlretrieve(url, pt)
print(f'Pretrained: {pt}  ({pt.stat().st_size/1e6:.1f} MB)')

## 9. Train - `model.train()` eksplisit (mirror yolo12.ipynb hyperparams)

In [ ]:
import time
from ultralytics import YOLO

model = YOLO(MODEL_CFG)
try:
    model.load(PRETRAINED)
    print(f'Loaded pretrained: {PRETRAINED}')
except Exception as e:
    print(f'[warn] could not load pretrained: {e}')

# === HyperMIL: register callback BEFORE train (installs aux head + loss wrapper) ===
if USE_HYPERMIL:
    sys.path.insert(0, str(REPO_DIR))
    from afb_yolov13 import make_hypermil_callback
    model.add_callback(
        'on_pretrain_routine_start',
        make_hypermil_callback(mil_weight=MIL_WEIGHT, mil_hidden=MIL_HIDDEN,
                               consist_weight=CONSIST_WEIGHT),
    )
    print(f'HyperMIL callback registered (mil_weight={MIL_WEIGHT}, '
          f'mil_hidden={MIL_HIDDEN}, consist_weight={CONSIST_WEIGHT})')

t0 = time.time()
results = model.train(
    # data + scale
    data=DATA_YAML,
    freeze=2,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    # optimizer
    optimizer='SGD',
    lr0=0.01, lrf=0.01,
    momentum=0.937, weight_decay=0.0005,
    cos_lr=True,
    # augmentation (mirror yolo12.ipynb)
    close_mosaic=10,
    hsv_h=0.1, hsv_s=0.3, hsv_v=0.3,
    degrees=30, translate=0.05, scale=0.1,
    flipud=0.3,
    mosaic=0.2, mixup=0.2,
    # control
    patience=0,
    amp=True,
    deterministic=True,
    seed=SEED,
    workers=8,
    # output
    project=RUN_PROJECT,
    name=f'{RUN_NAME}_train',
    exist_ok=True, save=True, verbose=True,
)
train_secs = time.time() - t0
print(f'\nTrain time: {train_secs/60:.1f} min')
print(f'Save dir  : {results.save_dir}')

# HyperMIL stats (if active)
if USE_HYPERMIL and hasattr(model.model, '_last_mil_loss'):
    print(f'\n=== HyperMIL final-batch stats ===')
    print(f'  MIL loss (last batch)      : {model.model._last_mil_loss:.4f}')
    print(f'  Predicted count mean       : {model.model._last_mil_count_mean:.2f}')
    print(f'  Target count mean          : {model.model._last_mil_target_mean:.2f}')

## 10. Log per-epoch curves ke W&B (dari `results.csv`)

In [ ]:
import pandas as pd

wandb.define_metric('epoch')
for k in [
    'train/box_loss', 'train/cls_loss', 'train/dfl_loss', 'train/total_loss',
    'val/box_loss', 'val/cls_loss', 'val/dfl_loss', 'val/total_loss',
    'val/mAP50', 'val/mAP50-95', 'val/precision', 'val/recall', 'lr/pg0',
]:
    wandb.define_metric(k, step_metric='epoch')

csv_path = Path(results.save_dir) / 'results.csv'
if csv_path.exists():
    df = pd.read_csv(csv_path); df.columns = [c.strip() for c in df.columns]
    col_map = [
        ('train/box_loss', 'train/box_loss'),
        ('train/cls_loss', 'train/cls_loss'),
        ('train/dfl_loss', 'train/dfl_loss'),
        ('val/box_loss', 'val/box_loss'),
        ('val/cls_loss', 'val/cls_loss'),
        ('val/dfl_loss', 'val/dfl_loss'),
        ('metrics/mAP50(B)', 'val/mAP50'),
        ('metrics/mAP50-95(B)', 'val/mAP50-95'),
        ('metrics/precision(B)', 'val/precision'),
        ('metrics/recall(B)', 'val/recall'),
        ('lr/pg0', 'lr/pg0'),
    ]
    for _, row in df.iterrows():
        try: ep = int(row.get('epoch', 0))
        except Exception: continue
        log = {'epoch': ep}
        for src, dst in col_map:
            if src in df.columns:
                try: log[dst] = float(row[src])
                except Exception: pass
        tb, tc, td = log.get('train/box_loss'), log.get('train/cls_loss'), log.get('train/dfl_loss')
        if None not in (tb, tc, td): log['train/total_loss'] = tb + tc + td
        vb, vc, vd = log.get('val/box_loss'), log.get('val/cls_loss'), log.get('val/dfl_loss')
        if None not in (vb, vc, vd): log['val/total_loss'] = vb + vc + vd
        run.log(log)
    print('Logged per-epoch curves to W&B.')
else:
    print('results.csv not found at', csv_path)

## 11. Test eval di holdout 101-image split

In [ ]:
best_pt = Path(results.save_dir) / 'weights' / 'best.pt'
print('Best ckpt:', best_pt)

eval_model = YOLO(str(best_pt))
eva = eval_model.val(data=DATA_YAML, split='test', imgsz=IMGSZ, device=DEVICE, verbose=False)

map50   = float(eva.box.map50)
map5095 = float(eva.box.map)
precision = float(np.mean(np.atleast_1d(eva.box.p)))
recall    = float(np.mean(np.atleast_1d(eva.box.r)))

# mAP@IoU=0.9 (index 8 dari [0.5, 0.55, ..., 0.95])
map_at_09 = float('nan')
try:
    ap_all = eva.box.all_ap
    if ap_all is not None and len(ap_all):
        ap = ap_all.mean(axis=0) if (hasattr(ap_all, 'ndim') and ap_all.ndim == 2) else ap_all
        if len(ap) >= 9: map_at_09 = float(ap[8])
except Exception as e:
    print(f'  (mAP@0.9 extract failed: {e})')

print(f'\n=== TEST RESULTS ({RUN_NAME}) ===')
print(f'  mAP50     : {map50:.4f}')
print(f'  mAP50-95  : {map5095:.4f}')
print(f'  mAP@0.9   : {map_at_09:.4f}')
print(f'  precision : {precision:.4f}')
print(f'  recall    : {recall:.4f}')
print(f'  train_min : {train_secs/60:.1f}')

run.summary['test/mAP50']     = map50
run.summary['test/mAP50-95']  = map5095
run.summary['test/mAP@0.9']   = map_at_09
run.summary['test/precision'] = precision
run.summary['test/recall']    = recall
run.summary['train/time_min'] = train_secs / 60

# Upload plots
for img in Path(results.save_dir).glob('*.png'):
    if any(t in img.stem.lower() for t in ('results', 'confusion', 'f1_curve', 'pr_curve', 'p_curve', 'r_curve')):
        try: run.log({f'plots/{img.stem}': wandb.Image(str(img))})
        except Exception: pass

run.finish()
print('\nW&B run finalised:', RUN_NAME)

## 12. Quick predict sample

In [ ]:
preds = eval_model.predict(
    source=f'{SPLIT_DIR}/test/images',
    save=True, imgsz=IMGSZ, conf=0.25, device=DEVICE,
)
print('Predictions saved to:', preds[0].save_dir if preds else None)

## 13. Diagnose baseline (CLI script call)

Output: per-IoU mAP, FP composition, FullPAD gate, HyperACE magnitude, recommendation. JSON disimpan untuk reference berikutnya.

In [ ]:
DIAG_OUT = (Path('/content') if IS_COLAB else REPO_DIR) / f'diag_{RUN_NAME}_val'
cmd = (
    f'python "{REPO_DIR}/scripts/diagnose_baseline.py" '
    f'--ckpt "{best_pt}" '
    f'--data "{DATA_YAML}" '
    f'--split val --imgsz {IMGSZ} --device {DEVICE} '
    f'--out "{DIAG_OUT}"'
)
print(cmd, '\n')
os.system(cmd)

import json
js = DIAG_OUT / 'diagnose.json'
if js.exists():
    s = json.loads(js.read_text())
    print('\n=== Recommendations ===')
    for r in s['recommendations']:
        print(f'  [{r["severity"]:>6}] {r["tag"]}: {r["reason"]}')

## 14. Inline probe - FullPAD_Tunnel gates + HyperACE magnitude

Verifikasi langsung apakah pathway HyperACE+FullPAD aktif setelah training:
- `gate ~= 0` -> alpha-trap, pathway tidak kontribusi -> sinyal arsitektur improvement (replace scalar gate, atau init lebih tinggi).
- `gate aktif (|g| > 0.05)` -> HyperACE memang dipakai, novelty arsitektur bisa fokus ke mekanisme di dalamnya.

In [ ]:
from ultralytics.nn.modules.block import FullPAD_Tunnel, HyperACE

probe_model = YOLO(str(best_pt))
m = probe_model.model.cuda().eval()

print('\n=== FullPAD_Tunnel gate values ===')
gates = []
for mod in m.modules():
    if isinstance(mod, FullPAD_Tunnel):
        g = mod.gate.detach().cpu().item()
        gates.append(g)
        status = 'ACTIVE' if abs(g) > 0.05 else ('marginal' if abs(g) > 0.01 else 'DEAD (alpha-trap)')
        print(f'  FullPAD #{len(gates):2d}  gate = {g:+.6f}   [{status}]')
if gates:
    print(f'\n  Mean |gate|: {sum(abs(g) for g in gates)/len(gates):.6f}')
    print(f'  Dead gates : {sum(1 for g in gates if abs(g)<0.01)}/{len(gates)}')

# HyperACE output magnitude
print('\n=== HyperACE output magnitude ===')
hyperace_outs = {}
handles = []
def make_hook(name):
    def fn(module, inp, out):
        hyperace_outs[name] = out.detach().abs().mean().item()
    return fn
for i, mod in enumerate(m.model):
    if isinstance(mod, HyperACE):
        handles.append(mod.register_forward_hook(make_hook(f'layer{i}')))
dummy = torch.randn(1, 3, IMGSZ, IMGSZ).cuda()
with torch.no_grad():
    _ = m(dummy)
for h in handles: h.remove()
for k, v in hyperace_outs.items():
    print(f'  {k}: |output|_mean = {v:.4e}')

## 15. Label quality probe (high-conf FP visual judgment)

Hipotesis: pada dataset AFB phone-camera, mungkin ada **bacilli yang GT miss-label** (Makerere annotation tidak 100% complete). Kalau benar, **model bisa correct tapi disebut FP** -> mAP50 ceiling artifisial.

Output: 30 crop high-conf FP yang jauh dari semua GT. Lo manual judge -> hitung % REAL_BACILLI.
- `> 50% REAL` -> label noise = ceiling -> paper pivot ke 'label quality study' atau pakai dataset lain.
- `20-50% REAL` -> campuran, masih bisa argue mAP50 underestimate.
- `< 20% REAL` -> model genuinely confuses smear/debris -> arch lift mustahil di dataset ini, reframe ke recall.

In [ ]:
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

VAL_IMG_DIR = Path(SPLIT_DIR) / 'val' / 'images'
VAL_LBL_DIR = Path(SPLIT_DIR) / 'val' / 'labels'
OUT_DIR     = (Path('/content') if IS_COLAB else REPO_DIR) / 'label_quality_probe'
OUT_DIR.mkdir(parents=True, exist_ok=True)

CONF_HIGH = 0.5
DIST_FAR  = 2.0
N_INSPECT = 30

high_conf_fps = []
for img_path in sorted(VAL_IMG_DIR.glob('*.jpg')):
    pil = Image.open(img_path).convert('RGB')
    W, H = pil.size
    res = eval_model.predict(str(img_path), conf=CONF_HIGH, iou=0.6, verbose=False, device=DEVICE)[0]
    if not len(res.boxes):
        continue
    pred = res.boxes.xyxy.cpu().numpy()
    pred_conf = res.boxes.conf.cpu().numpy()

    lp = VAL_LBL_DIR / (img_path.stem + '.txt')
    gt_centers, gt_diams = [], []
    if lp.exists():
        for ln in lp.read_text().strip().splitlines():
            parts = ln.split()
            if len(parts) >= 5:
                _, cx, cy, w, h = map(float, parts[:5])
                gt_centers.append([cx*W, cy*H])
                gt_diams.append(np.sqrt((w*W)*(h*H)))
    gt_centers = np.array(gt_centers) if gt_centers else np.empty((0,2))
    gt_diams   = np.array(gt_diams)   if gt_diams   else np.empty(0)

    for i, (x1,y1,x2,y2) in enumerate(pred):
        pc = np.array([(x1+x2)/2, (y1+y2)/2])
        if len(gt_centers) == 0:
            d_norm = float('inf')
        else:
            d = np.linalg.norm(gt_centers - pc, axis=1)
            j = d.argmin()
            d_norm = float(d[j] / max(gt_diams[j], 1))
        if d_norm > DIST_FAR:
            high_conf_fps.append(dict(img=img_path.name, box=(int(x1),int(y1),int(x2),int(y2)),
                                     conf=float(pred_conf[i]), dist=d_norm))

high_conf_fps.sort(key=lambda x: -x['conf'])
print(f'Total high-conf hard-neg FPs: {len(high_conf_fps)}')

n = min(N_INSPECT, len(high_conf_fps))
cols, rows = 6, (n + 5) // 6
fig, axes = plt.subplots(rows, cols, figsize=(cols*3, rows*3))
axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]
for i, fp in enumerate(high_conf_fps[:n]):
    pil = Image.open(VAL_IMG_DIR / fp['img']).convert('RGB')
    W, H = pil.size
    x1,y1,x2,y2 = fp['box']
    pad = 40
    cx1, cy1 = max(0, x1-pad), max(0, y1-pad)
    cx2, cy2 = min(W, x2+pad), min(H, y2+pad)
    crop = pil.crop((cx1, cy1, cx2, cy2)).copy()
    draw = ImageDraw.Draw(crop)
    draw.rectangle([x1-cx1, y1-cy1, x2-cx1, y2-cy1], outline='red', width=2)
    axes[i].imshow(crop)
    axes[i].set_title(f'#{i+1} conf={fp["conf"]:.2f}\n{fp["img"][:18]}', fontsize=7)
    axes[i].axis('off')
for ax in axes[n:]:
    ax.axis('off')
plt.tight_layout()
plt.savefig(OUT_DIR / 'top_high_conf_fps.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'\nSaved: {OUT_DIR / "top_high_conf_fps.png"}')
print('\nManual judgment - hitung % REAL_BACILLI / 30 -> kasih tau angkanya.')